# Demo de Text-to-Pandas con Gemini y CSV

Este notebook responde preguntas en lenguaje natural sobre el archivo
`datos/Base Haciendas Depurada.csv` usando Gemini AI y pandas.

**Estrategia:** Gemini genera código Python/pandas (en vez de SQL) que opera
sobre un DataFrame cargado en memoria. El flujo es:
`pregunta → código pandas → exec() → resultado`

In [11]:
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [12]:
# Cargar variables de entorno
load_dotenv()
print("Variables de entorno cargadas.")

Variables de entorno cargadas.


In [13]:
# Cargar el CSV en un DataFrame
df = pd.read_csv(
    "datos/Base Haciendas Depurada.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)
print(f"CSV cargado: {df.shape[0]} filas × {df.shape[1]} columnas")

CSV cargado: 2904 filas × 53 columnas


In [14]:
# Cargar el esquema de la base de datos
with open("SCHEMA.md", "r") as f:
    schema = f.read()
print("Esquema cargado:")
print(schema[:500] + "..." if len(schema) > 500 else schema)

Esquema cargado:
| # | Columna | Descripción |
| :--- | :--- | :--- |
| 1 | **FECHA** | Fecha del registro de la actividad. |
| 2 | **Semana** | Número de semana del año. |
| 3 | **Zona** | Región geográfica donde se ubica la hacienda. |
| 4 | **Unidad** | Código identificador de la hacienda. |
| 5 | **Nombre_Unidad** | Nombre de la hacienda. |
| 6 | **Real** | Indicador de rendimiento o ratio de producción real. |
| 7 | **Costo_Ha** | Costo total acumulado por hectárea. |
| 8 | **Atencion_Plantacion** | Costos ...


In [15]:
# Inicializar cliente Gemini
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Cliente Gemini inicializado.")

Cliente Gemini inicializado.


In [ ]:
# Pipeline: descomposición en tareas → ejecución tarea por tarea

_TASK_DECOMPOSITION_PROMPT = """You are a senior data analyst expert in agricultural production costs.
Given the description of a DataFrame called `df`, decompose the user's question into
an ordered list of concrete analytical tasks needed to answer it.

Output format — return ONLY the numbered list in Spanish, one task per line:
1. <task description>
2. <task description>
...

Rules:
- Each task must be a single, specific computation step.
- Tasks must be ordered so earlier results feed into later ones.
- The last task must produce the final answer.
- Do NOT write any code.

DataFrame schema:
{schema}
"""

_TASK_CODE_PROMPT = """You are a Python/pandas expert.
Write Python code to accomplish ONLY this specific task:

  Task: {task}

Available variables you can use:
- df: the original DataFrame (do NOT modify it)
- pd: pandas
{context_desc}

Rules:
- Return ONLY the Python code, no explanation, no markdown fences.
- Use only columns that exist in the schema.
- Store the result of this task in a variable called `task_result`.
- Do NOT import anything.

DataFrame schema:
{schema}
"""

def decompose_into_tasks(schema: str, question: str) -> list[str]:
    """Calls the reasoning model and returns a list of task strings."""
    response = client.models.generate_content(
        model=os.getenv("GEMINI_MODEL_REASONING", "gemini-2.5-pro"),
        config=types.GenerateContentConfig(
            system_instruction=_TASK_DECOMPOSITION_PROMPT.format(schema=schema)
        ),
        contents=question,
    )
    lines = response.text.strip().splitlines()
    tasks = []
    for line in lines:
        line = line.strip()
        if line and line[0].isdigit():
            # strip leading "1. ", "2. " etc.
            task = line.split(".", 1)[-1].strip()
            tasks.append(task)
    return tasks

def describe_context(context: dict) -> str:
    """Returns a human-readable description of accumulated task results."""
    if not context:
        return "(no previous results yet)"
    lines = []
    for var, val in context.items():
        if hasattr(val, "shape"):
            lines.append(f"- {var}: DataFrame/Series with shape {val.shape}")
            if hasattr(val, "columns"):
                lines.append(f"  columns: {list(val.columns)}")
        else:
            lines.append(f"- {var}: {type(val).__name__} = {str(val)[:100]}")
    return "\n".join(lines)

def generate_task_code(schema: str, task: str, context: dict) -> str:
    """Calls the code model to generate code for a single task."""
    response = client.models.generate_content(
        model=os.getenv("GEMINI_MODEL_CODE", "gemini-2.5-flash"),
        config=types.GenerateContentConfig(
            system_instruction=_TASK_CODE_PROMPT.format(
                task=task,
                schema=schema,
                context_desc=describe_context(context),
            )
        ),
        contents=task,
    )
    code = response.text.strip()
    if code.startswith("```"):
        code = code.split("\n", 1)[-1]
        code = code.rsplit("```", 1)[0].strip()
    return code

print("Funciones del pipeline de tareas definidas.")

In [17]:
import re
import difflib

def fix_column_names(code: str, columns: list) -> str:
    """Reemplaza strings entre comillas que no sean columnas reales pero se parezcan (fuzzy match)."""
    def replace_match(m):
        quote, name = m.group(1), m.group(2)
        if name in columns:
            return m.group(0)
        matches = difflib.get_close_matches(name, columns, n=1, cutoff=0.8)
        if matches:
            print(f"  [corrección] '{name}' → '{matches[0]}'")
            return f"{quote}{matches[0]}{quote}"
        return m.group(0)
    return re.sub(r"(['\"])([A-Za-z_][A-Za-z0-9_]*)\1", replace_match, code)

def run_code(df: pd.DataFrame, code: str) -> pd.DataFrame:
    """Corrige nombres de columnas, ejecuta el código y retorna `result`."""
    code = fix_column_names(code, df.columns.tolist())
    lines = code.strip().splitlines()
    if lines and "result" not in code:
        lines[-1] = f"result = {lines[-1].strip()}"
        code = "\n".join(lines)
    local_vars = {"df": df, "pd": pd}
    exec(code, {}, local_vars)
    return local_vars["result"]

print("Funciones fix_column_names y run_code definidas.")

Funciones fix_column_names y run_code definidas.


In [18]:
print("Todo listo. Usa ask('tu pregunta') para consultar el DataFrame.")

Todo listo. Usa ask('tu pregunta') para consultar el DataFrame.


In [ ]:
# Función principal: ejecuta el pipeline tarea por tarea
def ask(question: str):
    # Step 1: descomponer en tareas
    tasks = decompose_into_tasks(schema, question)
    print(f"Plan de tareas ({len(tasks)} pasos):")
    for i, t in enumerate(tasks, 1):
        print(f"  {i}. {t}")
    print()

    context = {}   # resultados acumulados de tareas anteriores
    last_result = None

    for i, task in enumerate(tasks, 1):
        print(f"--- Tarea {i}: {task} ---")

        # generar código solo para esta tarea
        code = generate_task_code(schema, task, context)
        code = fix_column_names(code, df.columns.tolist())
        print(f"Código:\n{code}\n")

        # ejecutar con df + resultados previos disponibles
        local_vars = {"df": df, "pd": pd, **context}
        try:
            exec(code, {}, local_vars)
        except Exception as e:
            raise RuntimeError(f"Error en tarea {i} ('{task}'): {e}") from e

        task_result = local_vars.get("task_result")
        context[f"task_{i}_result"] = task_result
        last_result = task_result

    return tasks, last_result

print("Función ask() definida.")

In [20]:
preguntas = [
    "Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?",
    "Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?",
    "¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?",
    # "¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?",
    # "¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?",
    # "¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?",
    # "¿Qué variables administrativas, pueden estar afectando los costos?",
    # "¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?",
    # "¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?",
    # "¿Qué programa se debería de ajustar para mejorar la productividad?",
]

for i, question in enumerate(preguntas, 1):
    print(f"{'='*80}")
    print(f"Pregunta {i}: {question}")
    print(f"{'='*80}")
    try:
        reasoning, code, result = ask(question)
        print("Resultado:")
        print(result.to_string())
        print(f"\n({len(result)} filas)\n")
    except Exception as e:
        print(f"ERROR: {e}\n")

Pregunta 1: Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?
Plan de tareas:
 1. Calcular el costo total para cada registro multiplicando la columna `Costo_Ha` por la columna `Total_Hectareas`.
2. Sumar los valores de tres columnas para todo el conjunto de datos: el costo total (calculado en el paso 1), `Total_Cajas` y `Total_Hectareas` para obtener los totales agregados.
3. Calcular el costo por caja dividiendo la suma de los costos totales entre la suma de `Total_Cajas`.
4. Calcular el costo por hectárea dividiendo la suma de los costos totales entre la suma de `Total_Hectareas`. 

Código generado:
 # Task 1: Calcular el costo total para cada registro
df_step1 = df.copy()
df_step1['Costo_Total_Registro'] = df_step1['Costo_Ha'] * df_step1['Total_Hectareas']

# Task 2: Sumar los valores de tres columnas para todo el conjunto de datos
total_costo_total = df_step1['Costo_Total_Registro'].sum()
total_cajas =

In [ ]:
print("Listo. El DataFrame `df` sigue disponible para más consultas.")